# German ASR Pipeline - Model Comparison

This notebook visualizes benchmark results and compares model performance.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

from asr_pipeline.utils import load_manifest

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## Load Benchmark Results

In [ ]:
# Load benchmark results
benchmark_path = '../artifacts/benchmark_results.parquet'

if Path(benchmark_path).exists():
    results_df = load_manifest(benchmark_path)
    print(f"Loaded {len(results_df)} benchmark records")
    print(f"\nModels: {results_df['model'].unique()}")
else:
    print(f"Benchmark results not found at {benchmark_path}")
    print("Please run the baseline benchmark first.")

## Model Summary

In [ ]:
if 'results_df' in locals():
    # Create summary
    summary = []
    for model in results_df['model'].unique():
        model_df = results_df[results_df['model'] == model]
        first_row = model_df.iloc[0]
        
        summary.append({
            'Model': model,
            'WER (%)': f"{first_row.get('wer', 0):.2f}",
            'CER (%)': f"{first_row.get('cer', 0):.2f}",
            'RTF': f"{first_row.get('rtf', 0):.3f}",
            'Samples/sec': f"{first_row.get('samples_per_second', 0):.1f}",
        })
    
    summary_df = pd.DataFrame(summary)
    summary_df = summary_df.sort_values('WER (%)')
    
    print("\nModel Comparison Summary:")
    print("=" * 80)
    print(summary_df.to_string(index=False))

## WER Comparison

In [ ]:
if 'results_df' in locals():
    # Get first row per model for metrics
    summary_rows = []
    for model in results_df['model'].unique():
        model_df = results_df[results_df['model'] == model]
        summary_rows.append(model_df.iloc[0])
    
    summary_metrics = pd.DataFrame(summary_rows)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # WER comparison
    sns.barplot(data=summary_metrics, x='model', y='wer', ax=axes[0], palette='viridis')
    axes[0].set_xlabel('Model')
    axes[0].set_ylabel('WER (%)')
    axes[0].set_title('Word Error Rate Comparison')
    axes[0].tick_params(axis='x', rotation=45)
    
    # CER comparison
    sns.barplot(data=summary_metrics, x='model', y='cer', ax=axes[1], palette='viridis')
    axes[1].set_xlabel('Model')
    axes[1].set_ylabel('CER (%)')
    axes[1].set_title('Character Error Rate Comparison')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('../artifacts/reports/wer_cer_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## Speed Comparison

In [ ]:
if 'summary_metrics' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # RTF comparison (lower is better)
    sns.barplot(data=summary_metrics, x='model', y='rtf', ax=axes[0], palette='plasma')
    axes[0].set_xlabel('Model')
    axes[0].set_ylabel('RTF (lower is better)')
    axes[0].set_title('Real-Time Factor Comparison')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].axhline(y=1.0, color='r', linestyle='--', label='Real-time')
    axes[0].legend()
    
    # Throughput comparison
    sns.barplot(data=summary_metrics, x='model', y='samples_per_second', ax=axes[1], palette='plasma')
    axes[1].set_xlabel('Model')
    axes[1].set_ylabel('Samples/sec')
    axes[1].set_title('Throughput Comparison')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('../artifacts/reports/speed_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## Accuracy vs Speed Trade-off

In [ ]:
if 'summary_metrics' in locals():
    fig, ax = plt.subplots(figsize=(10, 6))
    
    scatter = ax.scatter(
        summary_metrics['rtf'],
        summary_metrics['wer'],
        s=200,
        alpha=0.6,
        c=range(len(summary_metrics)),
        cmap='viridis'
    )
    
    # Add model labels
    for idx, row in summary_metrics.iterrows():
        ax.annotate(
            row['model'],
            (row['rtf'], row['wer']),
            xytext=(5, 5),
            textcoords='offset points',
            fontsize=9
        )
    
    ax.set_xlabel('RTF (lower is better)')
    ax.set_ylabel('WER (%) (lower is better)')
    ax.set_title('Accuracy vs Speed Trade-off')
    ax.axhline(y=summary_metrics['wer'].min(), color='r', linestyle='--', alpha=0.3)
    ax.axvline(x=summary_metrics['rtf'].min(), color='r', linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../artifacts/reports/accuracy_speed_tradeoff.png', dpi=150, bbox_inches='tight')
    plt.show()

## Load Model Selection

In [ ]:
# Load model selection results
selection_path = '../artifacts/model_selection.json'

if Path(selection_path).exists():
    with open(selection_path, 'r') as f:
        selection = json.load(f)
    
    print("Selected Models:")
    print("=" * 80)
    
    if selection.get('primary'):
        print(f"\n🥇 Primary: {selection['primary']['model']}")
        print(f"   WER: {selection['primary']['wer']:.2f}%")
        print(f"   CER: {selection['primary']['cer']:.2f}%")
        print(f"   RTF: {selection['primary']['rtf']:.3f}")
    
    if selection.get('fallback'):
        print(f"\n🥈 Fallback: {selection['fallback']['model']}")
        print(f"   WER: {selection['fallback']['wer']:.2f}%")
        print(f"   CER: {selection['fallback']['cer']:.2f}%")
        print(f"   RTF: {selection['fallback']['rtf']:.3f}")
    
    print(f"\nSelection Criteria:")
    criteria = selection.get('selection_criteria', {})
    print(f"   WER Weight: {criteria.get('wer_weight', 'N/A')}")
    print(f"   Speed Weight: {criteria.get('speed_weight', 'N/A')}")
else:
    print(f"Model selection results not found at {selection_path}")

## Error Analysis

In [ ]:
if 'results_df' in locals():
    # Analyze per-sample errors for each model
    for model in results_df['model'].unique()[:2]:  # First 2 models
        model_df = results_df[results_df['model'] == model]
        
        # Calculate per-sample WER
        import jiwer
        per_sample_wer = []
        for idx, row in model_df.iterrows():
            if row['reference'] and row['prediction']:
                wer = jiwer.wer([row['reference']], [row['prediction']]) * 100
                per_sample_wer.append(wer)
        
        fig, ax = plt.subplots(figsize=(10, 4))
        sns.histplot(per_sample_wer, bins=30, kde=True, ax=ax)
        ax.set_xlabel('Per-Sample WER (%)')
        ax.set_ylabel('Count')
        ax.set_title(f'Error Distribution - {model}')
        plt.tight_layout()
        plt.show()

## Export Comparison Report

In [ ]:
if 'summary_df' in locals():
    # Save comparison table
    output_path = '../artifacts/reports/model_comparison.csv'
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(output_path, index=False)
    print(f"Comparison report saved to {output_path}")